In [2]:
# Persistance in langgraph refers to ability to save and restore the state of a workflow over time.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [3]:
llm1 = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)
llm = ChatHuggingFace(llm=llm1)

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-y!",
 'explanation': 'Sure, let\'s break down this pizza joke for you:\n\nThe punchline, "Because it was feeling a little crust-y!" plays on two meanings of the word "crust." \n\n1. **Physical Crust**: Pizza has a baked outer edge called the "crust," which is the bread-like part at the rim of the pizza.\n2. **Slang Crust**: "Crusty" in this context is a play on words, referring to someone or something being grumpy, irritable, or in need of attention.\n\nSo, the joke is making a pun by suggesting that the pizza is feeling "crusty" in the sense that it might need some care or attention, much like a person who might go to a doctor when feeling unwell. The humor comes from the unexpected twist in the meaning of "crust" and the idea of a pizza, an inanimate object, visiting a doctor, which is something we don\'t typically associate with food.'}

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pasta'}, config=config1)

{'topic': 'pasta',
 'joke': "Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente and wasn't quite right without some sauce!",
 'explanation': 'This joke plays on the double meaning of "al dente," a term commonly used to describe pasta that is cooked until it\'s still slightly firm to the bite. The play on words is that the pasta, being "al dente," is not quite cooked properly, so it "feels a little al dente." The punchline then suggests that the pasta needs "some sauce," implying that it needs something to make it feel better or to be fully cooked and ready to eat, much like a person might need to see a doctor. It\'s a humorous take on a common culinary term!'}

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente and wasn't quite right without some sauce!", 'explanation': 'This joke plays on the double meaning of "al dente," a term commonly used to describe pasta that is cooked until it\'s still slightly firm to the bite. The play on words is that the pasta, being "al dente," is not quite cooked properly, so it "feels a little al dente." The punchline then suggests that the pasta needs "some sauce," implying that it needs something to make it feel better or to be fully cooked and ready to eat, much like a person might need to see a doctor. It\'s a humorous take on a common culinary term!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17af08-0906-6e89-8006-e3ccad574963'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-08T17:14:39.288487+00:00', parent_config={'configurable': {'thread_id': '1'

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente and wasn't quite right without some sauce!", 'explanation': 'This joke plays on the double meaning of "al dente," a term commonly used to describe pasta that is cooked until it\'s still slightly firm to the bite. The play on words is that the pasta, being "al dente," is not quite cooked properly, so it "feels a little al dente." The punchline then suggests that the pasta needs "some sauce," implying that it needs something to make it feel better or to be fully cooked and ready to eat, much like a person might need to see a doctor. It\'s a humorous take on a common culinary term!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17af08-0906-6e89-8006-e3ccad574963'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-08T17:14:39.288487+00:00', parent_config={'configurable': {'thread_id': '1

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'fries'}, config=config2)

{'topic': 'fries',
 'joke': 'Why did the fry go to school? \n\nTo get to the crisp level!',
 'explanation': 'This joke is a play on words and a bit of a pun. The setup, "Why did the fry go to school?" suggests a scenario about a fried food item, like a potato chip or a French fry, attending a place of learning. The punchline, "To get to the crisp level!" is a clever twist because "crisp" can refer both to the texture of a fried food (meaning well-cooked and crunchy) and to a level of achievement or proficiency in a subject.\n\nSo, the humor comes from the double meaning of "crisp" and the contrast between the idea of a food item going to school and the quality of its texture. It\'s a light-hearted, food-themed joke that plays with our expectations and puns on the word.'}

In [15]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'fries', 'joke': 'Why did the fry go to school? \n\nTo get to the crisp level!', 'explanation': 'This joke is a play on words and a bit of a pun. The setup, "Why did the fry go to school?" suggests a scenario about a fried food item, like a potato chip or a French fry, attending a place of learning. The punchline, "To get to the crisp level!" is a clever twist because "crisp" can refer both to the texture of a fried food (meaning well-cooked and crunchy) and to a level of achievement or proficiency in a subject.\n\nSo, the humor comes from the double meaning of "crisp" and the contrast between the idea of a food item going to school and the quality of its texture. It\'s a light-hearted, food-themed joke that plays with our expectations and puns on the word.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17af0f-582b-6aae-8002-d0f6570ae04d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at=

In [16]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'fries', 'joke': 'Why did the fry go to school? \n\nTo get to the crisp level!', 'explanation': 'This joke is a play on words and a bit of a pun. The setup, "Why did the fry go to school?" suggests a scenario about a fried food item, like a potato chip or a French fry, attending a place of learning. The punchline, "To get to the crisp level!" is a clever twist because "crisp" can refer both to the texture of a fried food (meaning well-cooked and crunchy) and to a level of achievement or proficiency in a subject.\n\nSo, the humor comes from the double meaning of "crisp" and the contrast between the idea of a food item going to school and the quality of its texture. It\'s a light-hearted, food-themed joke that plays with our expectations and puns on the word.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17af0f-582b-6aae-8002-d0f6570ae04d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at

Time Travel

In [25]:
workflow.get_state({"configurable": {"thread_id": "2", 'checkpoint_id': '1f17af0f-3b46-6914-8000-dd3ace19f93e'}})

StateSnapshot(values={'topic': 'fries'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f17af0f-3b46-6914-8000-dd3ace19f93e'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-08T17:17:52.462261+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17af0f-3b33-62b3-bfff-b39c5bf3f8b0'}}, tasks=(PregelTask(id='8000a7b8-6d34-e76b-ecd7-4c76f4659a45', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the fry go to school? \n\nTo get to the crisp level!'}),), interrupts=())

In [38]:
workflow.invoke(None, {"configurable": {"thread_id": "2", 'checkpoint_id': '1f17af0f-3b46-6914-8000-dd3ace19f93e'}})

{'topic': 'fries',
 'joke': 'Why did the fry go to school? \n\nTo become a french fry-iete!',
 'explanation': 'This joke plays on words and the concept of a profession or specialty. The setup asks "Why did the fry go to school?" which immediately makes us think of a fry in a cooking context. However, the punchline "To become a french fry-iete!" twists the concept by combining "french fry" with "-iete," a suffix often used to denote a field of study or profession. \n\nThe joke humorously suggests that just as someone might go to school to become a doctor, lawyer, or chef, a fry is going to school "to become a french fry-iete," implying a specialized field dedicated to being a French fry. This plays on our expectations and creates a silly, unexpected twist that is meant to be funny.'}

In [22]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'fries', 'joke': 'Why did the french fry go to the doctor?\n\nBecause it was feeling a little crispy on the outside, but kind of hollow on the inside!', 'explanation': 'This joke plays on the contrast between the physical characteristics of a French fry and the symptoms of a person experiencing emotional distress or a medical issue.\n\n1. **Crispy on the Outside, Hollow on the Inside**: \n   - **French Fry**: When a French fry is cooked, the outer layer gets crispy due to the dehydration and caramelization of the fats and sugars on the surface, while the inside can become slightly soft or hollow.\n   - **Person**: The phrase "crispy on the outside, but kind of hollow on the inside" metaphorically suggests that someone might appear tough or well-put-together on the surface but is struggling with internal issues. This could refer to emotional, psychological, or physical health problems.\n\n2. **Going to the Doctor**:\n   - **French Fry**: The French fry de

Updating State

In [36]:
workflow.update_state({"configurable": {"thread_id": "2", 'checkpoint_id': '1f17af0f-3b46-6914-8000-dd3ace19f93e', "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f17af4d-a95c-6366-8001-a521c5c0ab71'}}

In [37]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17af4d-a95c-6366-8001-a521c5c0ab71'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-08T17:45:48.305290+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17af0f-3b46-6914-8000-dd3ace19f93e'}}, tasks=(PregelTask(id='0c7ebe1d-380c-8127-6e1d-23ab82cae569', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17af47-1eb9-6c4f-8001-d0581a373b84'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-08T17:42:52.707131+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id

In [40]:
workflow.invoke(None, {"configurable": {"thread_id": "2", 'checkpoint_id': '1f17af47-1eb9-6c4f-8001-d0581a373b84'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa cross the road?\n\nTo get to the Bollywood party on the other side!',
 'explanation': 'This joke is a play on words and cultural references, blending humor with an understanding of popular Indian culture. The setup "Why did the samosa cross the road?" is a setup typically used for jokes, and the punchline "To get to the Bollywood party on the other side!" is the clever twist that makes it funny.\n\nHere\'s a breakdown of the joke:\n1. **Samosa**: A popular Indian savory snack, often enjoyed during parties and events.\n2. **Bollywood**: Refers to the Hindi language film industry based in Mumbai, India, known for extravagant and colorful parties.\n\nThe joke uses the samosa as the main character, giving it human-like intentions to cross the road to attend a Bollywood party. This imagery evokes the idea of a fun, celebratory atmosphere, and the humor comes from the unexpected combination of a snack food with the concept of attending a party

In [41]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa cross the road?\n\nTo get to the Bollywood party on the other side!', 'explanation': 'This joke is a play on words and cultural references, blending humor with an understanding of popular Indian culture. The setup "Why did the samosa cross the road?" is a setup typically used for jokes, and the punchline "To get to the Bollywood party on the other side!" is the clever twist that makes it funny.\n\nHere\'s a breakdown of the joke:\n1. **Samosa**: A popular Indian savory snack, often enjoyed during parties and events.\n2. **Bollywood**: Refers to the Hindi language film industry based in Mumbai, India, known for extravagant and colorful parties.\n\nThe joke uses the samosa as the main character, giving it human-like intentions to cross the road to attend a Bollywood party. This imagery evokes the idea of a fun, celebratory atmosphere, and the humor comes from the unexpected combination of a snack food with the concept 

Fault Tolerance

In [43]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [44]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [45]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))